# Lohnas 2025 Dataset Preparation

This notebook prepares the Lohnas 2025 dataset for use with the jaxcmr library.

## Experimental Design

Two experiments examining spaced repetition effects in free recall:

**Experiment 1 (Delayed Free Recall)**
- 184 subjects, 10 lists each
- List length: 20 words
- Post-study: 16s math distractor task
- 60s recall period

**Experiment 2 (Immediate Free Recall)**
- 156 subjects, 10 lists each  
- List length: 24 words
- No distractor (immediate recall)
- 60s recall period

**List Structure (both experiments)**
- 2 control lists: all unique items
- 8 critical lists: one repeated word per list
- Repeated item: first presentation at position 6 or 7, with lags of 5 or 6

## Output Format

Single HDF5 file `data/Lohnas2025.h5` with fields:
- `subject`: Subject ID (Exp2 subjects offset by 184)
- `experiment`: 1=delayed FR, 2=immediate FR
- `listLength`: 20 (Exp1) or 24 (Exp2)
- `list_type`: 1=control, 2=repeated
- `pres_itemnos`: Within-list item numbers (1-indexed, 0=padding)
- `pres_itemids`: Cross-list word pool IDs
- `recalls`: First study position of recalled items
- `rec_itemids`: Cross-list IDs of recalled items

## 1. Imports and Setup

In [ ]:
import csv
from collections import Counter
import numpy as np
from jaxcmr.helpers import save_dict_to_hdf5

## 2. Data Parsing Functions

In [ ]:
def parse_experiment_csv(filepath):
    """
    Parse a Lohnas experiment CSV file.
    
    Returns dict with keys: subjects, trials, pres_itemids, recalls_raw
    where each value is a list indexed by trial.
    """
    data = {
        'subjects': [],
        'trials': [],
        'pres_itemids': [],
        'recalls_raw': [],
    }
    
    # Read CSV and group by subject/trial
    trial_pres = {}  # (subject, trial) -> presentation sequence
    trial_rec = {}   # (subject, trial) -> recall sequence
    
    with open(filepath, 'r') as f:
        reader = csv.reader(f)
        next(reader)  # skip header
        
        for row in reader:
            subject = int(row[0])
            trial = int(row[1])
            pres0rec1 = int(row[2])
            items = [int(x) for x in row[3:] if x.strip()]
            
            key = (subject, trial)
            if pres0rec1 == 0:
                trial_pres[key] = items
            else:
                trial_rec[key] = items
    
    # Convert to ordered lists
    for key in sorted(trial_pres.keys()):
        subject, trial = key
        data['subjects'].append(subject)
        data['trials'].append(trial)
        data['pres_itemids'].append(trial_pres[key])
        data['recalls_raw'].append(trial_rec.get(key, []))
    
    return data

In [ ]:
def compute_pres_itemnos(pres_itemids):
    """
    Convert item IDs to within-list numbers.
    Repeated items get the same number (first occurrence index).
    Returns 1-indexed array.
    """
    seen = []
    itemnos = []
    for item_id in pres_itemids:
        if item_id not in seen:
            seen.append(item_id)
        itemnos.append(seen.index(item_id) + 1)  # 1-indexed
    return itemnos


def compute_recalls(pres_itemids, recalls_raw):
    """
    Map recalled item IDs to their first study position.
    Excludes intrusions (-1) and repeated recalls.
    Returns (recalls, rec_itemids) both 1-indexed with 0 padding.
    """
    recalls = []
    rec_itemids = []
    seen_positions = set()
    
    for rec_id in recalls_raw:
        if rec_id == -1:  # intrusion
            continue
        if rec_id not in pres_itemids:
            continue  # shouldn't happen, but safety check
        
        first_pos = pres_itemids.index(rec_id) + 1  # 1-indexed
        if first_pos in seen_positions:
            continue  # skip repeated recalls of same item
        
        seen_positions.add(first_pos)
        recalls.append(first_pos)
        rec_itemids.append(rec_id)
    
    return recalls, rec_itemids


def detect_list_type(pres_itemids):
    """
    Determine if list is control (1) or repeated (2).
    """
    counts = Counter(pres_itemids)
    return 2 if max(counts.values()) > 1 else 1

In [ ]:
def process_experiment(data, list_length, max_length, subject_offset=0, experiment_id=1):
    """
    Process parsed experiment data into arrays.
    
    Args:
        data: Output from parse_experiment_csv
        list_length: Actual list length for this experiment
        max_length: Maximum list length for padding
        subject_offset: Value to add to subject IDs
        experiment_id: Experiment identifier (1 or 2)
    
    Returns dict with arrays ready for HDF5.
    """
    n_trials = len(data['subjects'])
    
    result = {
        'subject': np.zeros((n_trials, 1), dtype='int64'),
        'experiment': np.full((n_trials, 1), experiment_id, dtype='int64'),
        'listLength': np.full((n_trials, 1), list_length, dtype='int64'),
        'list_type': np.zeros((n_trials, 1), dtype='int64'),
        'pres_itemnos': np.zeros((n_trials, max_length), dtype='int64'),
        'pres_itemids': np.zeros((n_trials, max_length), dtype='int64'),
        'recalls': np.zeros((n_trials, max_length), dtype='int64'),
        'rec_itemids': np.zeros((n_trials, max_length), dtype='int64'),
    }
    
    for i in range(n_trials):
        pres_ids = data['pres_itemids'][i]
        rec_raw = data['recalls_raw'][i]
        
        # Subject ID with offset
        result['subject'][i, 0] = data['subjects'][i] + subject_offset
        
        # List type
        result['list_type'][i, 0] = detect_list_type(pres_ids)
        
        # Presentation arrays
        pres_itemnos = compute_pres_itemnos(pres_ids)
        result['pres_itemnos'][i, :list_length] = pres_itemnos
        result['pres_itemids'][i, :list_length] = pres_ids
        
        # Recall arrays
        recalls, rec_itemids = compute_recalls(pres_ids, rec_raw)
        n_recalls = len(recalls)
        if n_recalls > 0:
            result['recalls'][i, :n_recalls] = recalls
            result['rec_itemids'][i, :n_recalls] = rec_itemids
    
    return result

## 3. Process Experiment 1 (Delayed Free Recall)

In [ ]:
exp1_raw = parse_experiment_csv('data/raw/Lohnas2025/Lohnas23_data_Exp1.csv')
print(f"Experiment 1: {len(exp1_raw['subjects'])} trials")
print(f"Subjects: {len(set(exp1_raw['subjects']))} unique")
print(f"List length: {len(exp1_raw['pres_itemids'][0])}")

In [ ]:
MAX_LENGTH = 24  # Maximum list length across both experiments

exp1_data = process_experiment(
    exp1_raw,
    list_length=20,
    max_length=MAX_LENGTH,
    subject_offset=0,
    experiment_id=1
)

print("Experiment 1 arrays:")
for k, v in exp1_data.items():
    print(f"  {k}: {v.shape}")

## 4. Process Experiment 2 (Immediate Free Recall)

In [ ]:
exp2_raw = parse_experiment_csv('data/raw/Lohnas2025/Lohnas23_data_Exp2.csv')
print(f"Experiment 2: {len(exp2_raw['subjects'])} trials")
print(f"Subjects: {len(set(exp2_raw['subjects']))} unique")
print(f"List length: {len(exp2_raw['pres_itemids'][0])}")

In [ ]:
# Offset subject IDs by max subject in Exp1
exp1_max_subject = max(exp1_raw['subjects'])
print(f"Exp1 max subject: {exp1_max_subject}")

exp2_data = process_experiment(
    exp2_raw,
    list_length=24,
    max_length=MAX_LENGTH,
    subject_offset=exp1_max_subject,
    experiment_id=2
)

print("\nExperiment 2 arrays:")
for k, v in exp2_data.items():
    print(f"  {k}: {v.shape}")

## 5. Combine Experiments

In [ ]:
result = {}
for key in exp1_data.keys():
    result[key] = np.concatenate([exp1_data[key], exp2_data[key]], axis=0)

print("Combined dataset:")
for k, v in result.items():
    print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    print(f"    min={v.min()}, max={v.max()}")

## 6. Verification and Validation

In [ ]:
# 1. All entries are 2D
for key, val in result.items():
    assert val.ndim == 2, f"{key} must be 2D. Got shape={val.shape}."
print("[PASS] All entries are 2D")

# 2. Single-column fields have shape (n_trials, 1)
single_col_keys = ["subject", "experiment", "listLength", "list_type"]
for k in single_col_keys:
    assert result[k].shape[1] == 1, f"{k} must have shape (n_trials, 1). Got {result[k].shape}."
print("[PASS] Single-column fields have correct shape")

# 3. Multi-column fields have 24 columns
multi_col_keys = ["pres_itemnos", "pres_itemids", "recalls", "rec_itemids"]
for k in multi_col_keys:
    assert result[k].shape[1] == MAX_LENGTH, f"{k} must have {MAX_LENGTH} columns. Got {result[k].shape}."
print(f"[PASS] Multi-column fields have {MAX_LENGTH} columns")

# 4. No negative values
for key, val in result.items():
    min_val = val.min()
    assert min_val >= 0, f"{key} has negative values (min={min_val})."
print("[PASS] No negative values")

In [ ]:
# 5. Zero-padding aligned across recalls and rec_itemids
recalls_zeros = (result["recalls"] == 0)
rec_itemids_zeros = (result["rec_itemids"] == 0)
assert (recalls_zeros == rec_itemids_zeros).all(), \
    "Mismatch in zero indices between recalls and rec_itemids."
print("[PASS] Zero-padding aligned between recalls and rec_itemids")

In [ ]:
# 6. Subject counts
n_subjects = len(np.unique(result['subject']))
expected_subjects = 184 + 156  # Exp1 + Exp2
assert n_subjects == expected_subjects, f"Expected {expected_subjects} subjects, got {n_subjects}"
print(f"[PASS] Subject count: {n_subjects}")

# 7. Trial counts per subject (should be 10 each)
unique_subjects, trial_counts = np.unique(result['subject'], return_counts=True)
assert (trial_counts == 10).all(), f"Not all subjects have 10 trials: {np.unique(trial_counts)}"
print("[PASS] All subjects have 10 trials")

# 8. List type distribution (2 control + 8 repeated per subject)
for subj in unique_subjects:
    mask = result['subject'].flatten() == subj
    list_types = result['list_type'][mask].flatten()
    n_control = (list_types == 1).sum()
    n_repeated = (list_types == 2).sum()
    assert n_control == 2, f"Subject {subj}: expected 2 control, got {n_control}"
    assert n_repeated == 8, f"Subject {subj}: expected 8 repeated, got {n_repeated}"
print("[PASS] List type distribution: 2 control + 8 repeated per subject")

In [ ]:
# 9. Experiment separation
exp1_mask = result['experiment'].flatten() == 1
exp2_mask = result['experiment'].flatten() == 2

assert exp1_mask.sum() == 1840, f"Expected 1840 Exp1 trials, got {exp1_mask.sum()}"
assert exp2_mask.sum() == 1560, f"Expected 1560 Exp2 trials, got {exp2_mask.sum()}"

# List lengths by experiment
assert (result['listLength'][exp1_mask] == 20).all(), "Exp1 list lengths should be 20"
assert (result['listLength'][exp2_mask] == 24).all(), "Exp2 list lengths should be 24"
print("[PASS] Experiment separation and list lengths correct")

In [ ]:
# Display sample data
print("Sample trial (first row):")
for k, v in result.items():
    print(f"  {k}: {v[0]}")

## 7. Save to HDF5

In [ ]:
output_path = 'data/Lohnas2025.h5'
save_dict_to_hdf5(result, output_path)
print(f"Saved to {output_path}")

In [ ]:
# Verify saved file
import h5py

with h5py.File(output_path, 'r') as f:
    print("Saved file structure:")
    def print_structure(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: shape={obj.shape}, dtype={obj.dtype}")
    f.visititems(print_structure)